In [0]:
%sql
CREATE OR REPLACE TABLE banking.gold.customer_360 AS

WITH account_agg AS (
    SELECT
        customer_id,
        COUNT(account_id) AS total_accounts,
        SUM(balance) AS total_balance
    FROM banking.silver.accounts
    GROUP BY customer_id
),

txn_agg AS (
    SELECT
        a.customer_id,
        COUNT(t.txn_id) AS total_transactions,
        SUM(t.amount) AS total_transaction_amount
    FROM banking.silver.transactions t
    JOIN banking.silver.accounts a
        ON t.account_id = a.account_id
    GROUP BY a.customer_id
),

credit_latest AS (
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY bureau_pull_date DESC
            ) AS rn
        FROM banking.silver.credit_bureau_reports
    )
    WHERE rn = 1
)

SELECT
    c.customer_id,
    CONCAT(c.first_name,' ',c.last_name) AS customer_name,
    b.branch_name,
    COALESCE(aa.total_accounts,0) AS total_accounts,
    COALESCE(aa.total_balance,0) AS total_balance,
    COALESCE(ta.total_transactions,0) AS total_transactions,
    COALESCE(ta.total_transaction_amount,0) AS total_transaction_amount,
    cr.credit_score,
    cr.risk_grade,
    cr.external_active_loans,
    cr.external_overdue_amount,

    CASE
        WHEN aa.total_balance >= 500000 THEN 'HIGH_VALUE'
        WHEN aa.total_balance >= 100000 THEN 'MEDIUM_VALUE'
        ELSE 'LOW_VALUE'
    END AS customer_segment

FROM banking.silver.customers c

LEFT JOIN account_agg aa
    ON c.customer_id = aa.customer_id

LEFT JOIN txn_agg ta
    ON c.customer_id = ta.customer_id

LEFT JOIN banking.silver.branches b
    ON c.branch_code = b.branch_code

LEFT JOIN credit_latest cr
    ON c.customer_id = cr.customer_id


In [0]:
count = spark.sql("""
    SELECT 
        COUNT(*) AS cnt
    FROM banking.gold.customer_360
    """).collect()[0]["cnt"]

dbutils.notebook.exit(str(count))